# PaySim model experiments

Reproducible inspection of the full-data training results. Production logic lives in `src/`; this notebook reads the immutable metadata and comparison artifacts produced by the pipeline.

In [1]:
import json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
metadata = json.loads((ROOT / 'models/fraud_model_v1.0.0_metadata.json').read_text(encoding='utf-8'))
comparison = pd.read_csv(ROOT / 'output/evaluation/model_comparison.csv')
{'model_version': metadata['version'], 'selected_model': metadata['model_name'], 'threshold': metadata['threshold'], 'git_commit': metadata['git_commit']}

{'model_version': '1.0.0', 'selected_model': 'random_forest', 'threshold': 0.31999999999999995, 'git_commit': None}

## Validation comparison

All models use the complete validation period. Logistic Regression uses all train rows; Random Forest retains all fraud rows and caps the total fit set at 500,000.

In [2]:
columns = ['model','training_rows','pr_auc','precision','recall','f2','alert_count','fraud_amount_captured','fraud_amount_missed','fraud_amount_capture_rate']
comparison[columns].sort_values('f2', ascending=False)

,model,training_rows,pr_auc,precision,recall,f2,alert_count,fraud_amount_captured,fraud_amount_missed,fraud_amount_capture_rate
2,random_forest,500000,0.999997,0.996622,1.000000,0.999322,1184,1.497277e+09,0.000000e+00,1.000000
1,logistic_regression,6082007,0.846412,0.640127,0.851695,0.798887,1570,1.473381e+09,2.389562e+07,0.984041
0,baseline_rule,6082007,0.024149,0.030893,0.981356,0.137191,37484,1.330804e+09,1.664728e+08,0.888816


## Final validation and test metrics

The test set is used once with the threshold selected exclusively on validation.

In [3]:
metric_table = pd.DataFrame([metadata['validation_metrics'], metadata['test_metrics']], index=['validation','test'])
metric_table[['pr_auc','precision','recall','f2','tn','fp','fn','tp','alert_count','fraud_amount_captured','fraud_amount_missed','fraud_amount_capture_rate']]

,pr_auc,precision,recall,f2,tn,fp,fn,tp,alert_count,fraud_amount_captured,fraud_amount_missed,fraud_amount_capture_rate
validation,0.999997,0.996622,1.000000,0.999322,189963,4,0,1180,1184,1.497277e+09,0.00000,1.000000
test,0.999999,0.999201,0.999201,0.999201,88213,1,1,1251,1252,2.129730e+09,399045.09375,0.999813


In [4]:
pd.DataFrame({'feature': metadata['feature_list']}).assign(feature_count=len(metadata['feature_list']))

,feature,feature_count
0,TransactionType,21
1,Amount,21
2,LogAmount,21
3,StepRaw,21
4,Hour,21
5,Day,21
6,OldBalanceOrig,21
7,NewBalanceOrig,21
8,BalanceChangeOrig,21
9,BalanceDropOrig,21


## Visual evidence

![Validation precision-recall curve](../output/evaluation/precision_recall_validation.svg)

![Test confusion matrix](../output/evaluation/confusion_matrix_test.svg)

## Conclusion

Random Forest is selected because it has the highest validation F2 (0.999322), captures 100% of validation fraud amount and produces 1,184 validation alerts. On untouched test data it records one FP and one FN, with a 99.981267% fraud-amount capture rate. These unusually high PaySim results must not be generalized to production banking data.